# S24 · Simulating an (s, S) inventory policy

We run a shop that restocks with a simple rule: when stock falls to `s`, order
back up to `S`. We play it out day by day, draw the stock "sawtooth", measure what
it costs and how many customers it serves, and compare two settings. This is the
same idea as notebook 1: a system that hops between states (stock levels), watched
over the long run.

**New here? Read this once.**

- New to Python? Press play on each cell, top to bottom, and read the note above
  each one. You do not need to write any code to follow along.
- New to the idea? A shop's stock level is a system that hops step by step, exactly
  like the weather in notebook 1. See `primers/markov_chains.md` for the picture.
- Already confident? Look for the cell marked **Stretch (optional)** near the end.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there.

In [ ]:
# This notebook uses numpy and matplotlib.
# Google Colab already ships both, so there is nothing to install.
print("Setup complete - nothing to install.")

In [ ]:
import numpy as np                 # random daily demand and arrays
import matplotlib.pyplot as plt    # drawing charts

# A seed fixes the random demand so the notebook gives the same result every run.
np.random.seed(0)

print("Libraries ready.")

## Step 1 — the (s, S) rule in plain words

A shop holds stock of one product, say phone chargers. Each day a random number of
customers want one. The restock rule has just two numbers:

- **s** is the *reorder point*. When stock drops to `s` or below, we place an order.
- **S** is the *order-up-to level*. We order just enough to bring stock back up to `S`.

Two numbers fully describe the policy. Our whole job is to play it out and see how
well it does.

In [ ]:
# The two numbers that define the policy.
reorder_point = 20        # this is s: order when stock falls to here
order_up_to_level = 60    # this is S: refill back up to here

# The average number of units customers want per day.
average_daily_demand = 10

print("Policy: when stock <=", reorder_point, "order back up to", order_up_to_level)
print("Average daily demand:", average_daily_demand, "units")

## Step 2 — set the costs and how long to run

Every unit sitting in the warehouse costs a little each day (a **holding cost**).
Every unit a customer wanted but we could not supply costs more (a **stockout
penalty**). Running out should hurt more than holding a little extra, so we set the
stockout penalty higher.

In [ ]:
holding_cost_per_unit_per_day = 1.0    # cost of keeping one unit in stock for a day
stockout_penalty_per_unit = 5.0        # cost of failing to supply one unit of demand

number_of_days = 200                   # how long to play the shop out

print("Holding cost :", holding_cost_per_unit_per_day, "per unit per day")
print("Stockout cost:", stockout_penalty_per_unit, "per unit short")
print("Playing out", number_of_days, "days")

## Step 3 — write the day-by-day rule as a function

We put the whole daily loop in a **function** so we can reuse it for different
`(s, S)` settings without copying it. Read it slowly; each line is one small,
everyday step.

Each day the shop does six things: receive any order that arrived, meet the random
demand as far as stock allows, record a shortfall if demand ran past stock, charge
the holding cost on whatever is left, and finally place a new order if stock has
fallen to `s`.

In [ ]:
def simulate_inventory(reorder_point, order_up_to_level):
    """Play the (s, S) policy for number_of_days and return the results."""

    # Start with a full shelf.
    stock = order_up_to_level

    # An order takes 2 days to arrive (the "lead time").
    lead_time = 2
    # We remember an order on its way as a day it arrives and an amount.
    incoming_order_day = None
    incoming_order_amount = 0

    # Records we will fill in as the days go by.
    stock_each_day = []      # stock level at the start of each day
    total_holding_cost = 0.0
    total_stockout_cost = 0.0
    total_demand = 0         # how many units customers wanted in total
    total_units_short = 0    # how many units we could NOT supply

    for day in range(number_of_days):

        # 1) If a previous order arrives today, add it to the shelf.
        if incoming_order_day == day:
            stock = stock + incoming_order_amount
            incoming_order_day = None
            incoming_order_amount = 0

        # Record the stock at the start of the day (after any delivery).
        stock_each_day.append(stock)

        # 2) Draw today's random demand. Poisson is a common count distribution
        #    for "how many customers turned up".
        demand_today = np.random.poisson(average_daily_demand)
        total_demand = total_demand + demand_today

        # 3) Sell as much as we can. If demand is more than stock, we run short.
        if demand_today <= stock:
            stock = stock - demand_today
        else:
            units_short = demand_today - stock
            total_units_short = total_units_short + units_short
            total_stockout_cost = total_stockout_cost + units_short * stockout_penalty_per_unit
            stock = 0   # the shelf is now empty

        # 4) Charge the holding cost on whatever is left on the shelf overnight.
        total_holding_cost = total_holding_cost + stock * holding_cost_per_unit_per_day

        # 5) Reorder if stock has fallen to the reorder point and none is on the way.
        if stock <= reorder_point and incoming_order_day is None:
            incoming_order_amount = order_up_to_level - stock
            incoming_order_day = day + lead_time

    # Bundle the results into a dictionary so they are easy to read back.
    total_cost = total_holding_cost + total_stockout_cost
    units_supplied = total_demand - total_units_short
    service_level = units_supplied / total_demand   # fraction of demand we met

    results = {
        "stock_each_day": stock_each_day,
        "total_cost": total_cost,
        "total_holding_cost": total_holding_cost,
        "total_stockout_cost": total_stockout_cost,
        "service_level": service_level,
    }
    return results

print("The simulate_inventory function is defined.")

## Step 4 — run the shop once

We call the function with our chosen `s` and `S` and print the headline numbers a
manager would care about: the average cost per day, and the service level (the
fraction of customer demand we actually met).

In [ ]:
results = simulate_inventory(reorder_point, order_up_to_level)

average_cost_per_day = results["total_cost"] / number_of_days

print("Average cost per day :", round(average_cost_per_day, 2))
print("  from holding       :", round(results["total_holding_cost"] / number_of_days, 2), "per day")
print("  from stockouts     :", round(results["total_stockout_cost"] / number_of_days, 2), "per day")
print("Service level        :", round(results["service_level"] * 100, 1), "% of demand met")

## Step 5 — draw the stock sawtooth

Stock drifts down as customers buy, then jumps back up to `S` after each order
arrives. The dashed lines mark `s` (reorder point) and `S` (order-up-to level).
This zig-zag is the signature shape of inventory control, and it is the same
"hop step by step" picture as the weather chain, now on stock levels.

In [ ]:
stock_each_day = results["stock_each_day"]

plt.figure(figsize=(9, 5))
plt.step(range(number_of_days), stock_each_day, where="post",
         color="#2E75B6", label="stock on hand")

# Dashed reference lines for s and S.
plt.axhline(reorder_point, color="#C0392B", linestyle="--", label="reorder point s")
plt.axhline(order_up_to_level, color="#27AE60", linestyle="--", label="order-up-to S")

plt.xlabel("day")
plt.ylabel("units in stock")
plt.title("The (s, S) inventory sawtooth")
plt.legend()
plt.show()

## Step 6 — try a second, more cautious setting

Higher `s` and `S` means we keep more stock on the shelf. That should raise the
holding cost but improve the service level, because we run out less often. Let us
run a second policy and compare the two side by side.

In [ ]:
# A second policy that keeps more stock on the shelf.
reorder_point_B = 35
order_up_to_level_B = 90

results_A = results   # the first policy from Step 4
results_B = simulate_inventory(reorder_point_B, order_up_to_level_B)

cost_A = results_A["total_cost"] / number_of_days
cost_B = results_B["total_cost"] / number_of_days

print("Policy A  (s=20, S=60):  cost/day =", round(cost_A, 2),
      "  service =", round(results_A["service_level"] * 100, 1), "%")
print("Policy B  (s=35, S=90):  cost/day =", round(cost_B, 2),
      "  service =", round(results_B["service_level"] * 100, 1), "%")

## Step 7 — plot both stock levels together

Policy B keeps more stock, so it sits higher on the chart and rarely touches zero.
It serves more demand, but it pays for that with a higher holding cost. This
trade-off, more stock costs more but serves more, is the whole game.

In [ ]:
plt.figure(figsize=(9, 5))
plt.step(range(number_of_days), results_A["stock_each_day"], where="post",
         color="#2E75B6", label="Policy A (s=20, S=60)")
plt.step(range(number_of_days), results_B["stock_each_day"], where="post",
         color="#E67E22", label="Policy B (s=35, S=90)")
plt.axhline(0, color="grey", linewidth=0.8)
plt.xlabel("day")
plt.ylabel("units in stock")
plt.title("Two (s, S) policies compared")
plt.legend()
plt.show()

### Stretch (optional) — this shop is a Markov chain

Here is the link back to notebook 1, for anyone curious. The stock level each
morning is a **state**, just like Sunny or Rainy. Given the `(s, S)` rule and the
demand distribution, tomorrow's stock depends only on today's stock (plus a random
demand draw). That is exactly the Markov property, so the stock level is a Markov
chain hopping between levels. Its long-run mix, the **stationary distribution** of
stock levels, is what our sawtooth is sampling. We can estimate that long-run
spread just by counting how often each stock level shows up in a long run.

In [ ]:
# Play out a long run and see how often each stock level appears.
long_run = simulate_inventory(reorder_point, order_up_to_level)
stock_levels = np.array(long_run["stock_each_day"])

plt.figure(figsize=(8, 4))
plt.hist(stock_levels, bins=range(0, order_up_to_level + 2), color="#2E75B6")
plt.axvline(reorder_point, color="#C0392B", linestyle="--", label="reorder point s")
plt.xlabel("units in stock")
plt.ylabel("number of days at that level")
plt.title("Long-run spread of stock levels (the chain's stationary distribution)")
plt.legend()
plt.show()

print("Average stock held:", round(stock_levels.mean(), 1), "units")

## What you just did

You played an `(s, S)` inventory policy out day by day, drew the stock sawtooth,
and measured the two things a manager cares about: cost per day and service level.
Comparing two settings showed the trade-off, that holding more stock costs more but
serves more customers. In the next notebook we stop guessing settings and search
for a good one on purpose, using demand estimated from past sales.